# Asynchronous simulation for BMA

A short notebook exploring the dynamics of asynchronous networks

## load the libraries

In [1]:
import pybma

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import numpy as np

from copy import deepcopy

## load the model and check the stability

In [2]:
m = pybma.load_model("./ToyModelStable.json")
qn = pybma.model_to_qn(m)
p = pybma.check_stability(qn)
print(p)

{'ProofProgression': {'stable': True, 'history': [(4, {1: (0, 0), 2: (0, 0), 3: (0, 0)}), (3, {1: (0, 0), 2: (0, 0), 3: (0, 0)}), (2, {1: (0, 0), 2: (0, 0), 3: (0, 0)}), (1, {1: (0, 0), 2: (0, 0), 3: (0, 4)}), (0, {1: (0, 0), 2: (0, 4), 3: (0, 4)})]}, 'CounterExample': None}


In [3]:
final_state = dict( [ (varid,p['ProofProgression']['history'][0][1][varid][0]) for varid in p['ProofProgression']['history'][0][1].keys() ] )

In [4]:
# Some issue with number of steps; 3 gives us two states
pybma.simulate(qn,steps=3,initial_values=final_state)

{1: [0, 0], 2: [0, 0], 3: [0, 0]}

## Mutate the model

In [5]:
m['Model']['Variables'][0]['Formula'] = "2"
qM = pybma.model_to_qn(m)
pM = pybma.check_stability(qM)

print("###Modified Model###")
print(pM)

###Modified Model###
{'ProofProgression': {'stable': True, 'history': [(4, {1: (2, 2), 2: (2, 2), 3: (2, 2)}), (3, {1: (2, 2), 2: (2, 2), 3: (2, 2)}), (2, {1: (2, 2), 2: (2, 2), 3: (2, 2)}), (1, {1: (2, 2), 2: (2, 2), 3: (0, 4)}), (0, {1: (2, 2), 2: (0, 4), 3: (0, 4)})]}, 'CounterExample': None}


### Simulate the new model from the old stable state

In [6]:
pybma.simulate(qM,steps=3,initial_values=final_state)

{1: [0, 1], 2: [0, 0], 3: [0, 0]}

In [7]:
def stateToTuple(state):
    return tuple(state[k] for k in sorted(state.keys()))
    
def tupleToState(tupleState,variables):
    return dict(zip(sorted(variables), tupleState)) #return(dict(zip(variables, tupleState)))

def async_step(qn,state):
    variables = list(state.keys())
    sync_result = pybma.simulate(qn,steps=3,initial_values=state)
    #print(sync_result)
    #print(len(sync_result))
    sync_result_final = dict([(varid,sync_result[varid][-1]) for varid in variables])
    
    async_result = []
    for varid in variables:
        successor = deepcopy(state) # copy old state
        successor[varid] = sync_result_final[varid]
        async_result.append(stateToTuple(successor))
    async_result = set(async_result)
    return(async_result)

In [8]:
def check_index_mapping(variables: list, encoder: StateEncoder, i: int, j: int):
    print(f"Gene i={i} -> '{variables[i]}'")
    print(f"Gene j={j} -> '{variables[j]}'")
    print(f"Encoder max_levels: {encoder.max_levels}")
    print(f"variables list:     {variables}")
    # These must be in the same order or encode/decode will map wrong genes

In [9]:
async_step(qM,final_state)

{(0, 0, 0), (1, 0, 0)}

In [10]:
class StateEncoder:
    def __init__(self, model):
        maxes = [v['RangeTo'] for v in model['Model']['Variables'] ]
        variables = [v['Id'] for v in model['Model']['Variables'] ]
        max_levels = [k for k, v in sorted(zip(maxes, variables), key=lambda x: x[1])]
        self.max_levels = max_levels
        # Precompute strides for mixed-radix encoding
        self.strides = np.cumprod([1] + [m + 1 for m in max_levels[:-1]])

    def encode(self, state: tuple) -> int:
        return int(np.dot(state, self.strides))

    def decode(self, code: int) -> tuple:
        result = []
        for m in self.max_levels:
            result.append(code % (m + 1))
            code //= (m + 1)
        return tuple(result)

In [11]:
def async_simulate(qn,initial_values=None, encoder=None):
    if initial_values == None or encoder == None: raise
    
    variables = list(initial_values.keys())
    initial_state = stateToTuple(initial_values)
    
    graph: dict[int, set[int]] = {}

    # Adding a transition
    def add_transition(graph, from_state, to_state):
        graph.setdefault(encoder.encode(from_state), set()).add(encoder.encode(to_state))
    
    final_values = [initial_state]
    stop = False
    while not stop:
        stop = True
        updated_final_values = []
        for final_values_state in final_values:
            if encoder.encode(final_values_state) in graph.keys():
                continue
            stop = False
            final_values_single = tupleToState(final_values_state,variables)
            step_result = async_step(qn,final_values_single)
            updated_final_values.append(step_result)
            for successor in step_result:
                add_transition(graph,final_values_state,successor)
        final_values = [x for sublist in updated_final_values for x in sublist]  
    return(graph)

In [12]:
from collections import deque

def exists_order_before(
    graph: dict[int, set[int]],
    encoder: StateEncoder,
    start: tuple,
    i: int, v: int,
    j: int, w: int
) -> bool:
    """
    Returns True if there exists a path from start where
    gene i takes value v at some state BEFORE gene j takes value w.
    """
    start_code = encoder.encode(start)
    
    visited = set()
    queue = deque()
    queue.append((start_code, encoder.decode(start_code)[i] == v))
    
    while queue:
        code, i_seen = queue.popleft()
        
        if (code, i_seen) in visited:
            continue
        visited.add((code, i_seen))
        
        state = encoder.decode(code)
        
        if state[j] == w and not i_seen:
            continue
        
        if i_seen and state[j] != w:
            return True
        
        for successor_code in graph.get(code, set()):
            new_i_seen = i_seen or (encoder.decode(successor_code)[i] == v)
            queue.append((successor_code, new_i_seen))
    
    return False

In [13]:
encoder = StateEncoder(m)
graph = async_simulate(qM,final_state,encoder)
print(graph)

{0: {0, 1}, 1: {1, 2, 6}, 6: {31, 6, 7}, 2: {2, 7}, 7: {32, 12, 7}, 31: {32, 31}, 32: {32, 37}, 12: {12, 37}, 37: {37, 62}, 62: {62}}


In [14]:
variables = list(final_state.keys())

check_index_mapping(variables, encoder, 1, 2)

Gene i=1 -> '2'
Gene j=2 -> '3'
Encoder max_levels: [4, 4, 4]
variables list:     [1, 2, 3]


In [15]:
exists_order_before(graph, encoder, (0, 0, 0), 0, 1, 1, 1)

True

In [16]:
exists_order_before(graph, encoder, (0, 0, 0), 1, 1, 0, 1)

False

In [17]:
def exists_order_before_online(
    qn: BNModel,
    encoder: StateEncoder,
    variables: dict [int, int],
    start: tuple,
    i: int, v: int,
    j: int, w: int
) -> tuple[bool, list[tuple] | None]:
    """
    Returns (True, trace) if there exists a path from start where
    gene i takes value v before gene j takes value w, or (False, None).
    trace is a list of decoded state tuples from start to the witnessing state.
    """
    start_code = encoder.encode(start)
    initial_i_seen = start[i] == v
        
    visited = set()
    # parent maps (code, i_seen) -> (parent_code, parent_i_seen) | None
    parent: dict[tuple[int, bool], tuple[int, bool] | None] = {}

    queue = deque()
    queue.append((start_code, initial_i_seen))
    parent[(start_code, initial_i_seen)] = None

    while queue:
        code, i_seen = queue.popleft()

        if (code, i_seen) in visited:
            continue
        visited.add((code, i_seen))

        state = encoder.decode(code)

        if state[j] == w and not i_seen:
            continue

        if i_seen and state[j] != w:
            return True, _reconstruct_trace(parent, encoder, code, i_seen)

        #print("P:",state[i],state[j])
        for successor in async_step(qn, tupleToState(state, variables)):
            #print("S:",successor[i],successor[j])
            successor_code = encoder.encode(successor)
            new_i_seen = i_seen or (successor[i] == v)
            key = (successor_code, new_i_seen)
            if key not in parent:
                parent[key] = (code, i_seen)
            queue.append((successor_code, new_i_seen))

    return False, None


def _reconstruct_trace(
    parent: dict[tuple[int, bool], tuple[int, bool] | None],
    encoder: StateEncoder,
    code: int,
    i_seen: bool
) -> list[tuple]:
    """Walk parent pointers back to the start and reverse."""
    trace = []
    cursor: tuple[int, bool] | None = (code, i_seen)
    while cursor is not None:
        trace.append(encoder.decode(cursor[0]))
        cursor = parent[cursor]
    trace.reverse()
    return trace

In [18]:
variables = list(final_state.keys())
exists_order_before_online(qM, encoder, variables, (0, 0, 0), 0, 2, 1, 1)

(True, [(0, 0, 0), (1, 0, 0), (2, 0, 0)])

In [19]:
exists_order_before_online(qM, encoder, variables, (0, 0, 0), 1, 1, 0, 1)

(False, None)

## B cell differentiation

In [20]:
m = pybma.load_model("./B-cell-undifferentiated-stable.json")
qn = pybma.model_to_qn(m)
p = pybma.check_stability(qn)
print("Stable =", p['ProofProgression']['stable'])

Stable = True


In [21]:
final_state = dict( [ (varid,p['ProofProgression']['history'][0][1][varid][0]) for varid in p['ProofProgression']['history'][0][1].keys() ] )

In [22]:
variable_names_lookup = { v['Id']:v['Name'] for v in m['Model']['Variables'] }
variable_ids_lookup = { v['Name']:v['Id'] for v in m['Model']['Variables'] }
print('CLP',variable_ids_lookup['CLP'])
print('Naive B cell',variable_ids_lookup['Naive B cell'])

CLP 146
Naive B cell 164


In [23]:
print(final_state)

{2: 0, 3: 0, 7: 0, 8: 0, 11: 0, 17: 0, 19: 0, 30: 0, 38: 0, 59: 2, 68: 0, 74: 0, 120: 1, 121: 1, 122: 1, 126: 0, 130: 0, 133: 0, 146: 1, 147: 0, 148: 0, 149: 0, 164: 0, 171: 0, 179: 0, 180: 0, 181: 0, 182: 0, 198: 0, 206: 0, 211: 0, 231: 2, 232: 0, 236: 0, 237: 0, 247: 0, 254: 0, 257: 0, 266: 0, 272: 1, 275: 0, 280: 1, 293: 0, 307: 0, 313: 0, 322: 0, 331: 0, 343: 0, 344: 0, 366: 0, 383: 0, 390: 0, 392: 0, 408: 0, 414: 0, 415: 0, 423: 0, 424: 0, 425: 0, 437: 0, 457: 0, 464: 0, 471: 0, 472: 0, 473: 0}


In [24]:
print("CLP final =", final_state[variable_ids_lookup['CLP']])
print("Naive B cell final =", final_state[variable_ids_lookup['Naive B cell']])

CLP final = 1
Naive B cell final = 0


In [25]:
m['Model']['Variables'][0]

{'Name': 'Ilr7',
 'Id': 2,
 'RangeFrom': 0,
 'RangeTo': 2,
 'Formula': 'min(2,(2*var(7)))+var(74)+var(8)-(4*var(30))-max(0,(2*var(206)-var(211)))'}

In [26]:

varlist = [v['Name'] for v in m['Model']['Variables'] ]
variableDict = dict(zip(varlist,range(len(varlist))))

In [27]:
variableDict

{'Ilr7': 0,
 'Il7': 1,
 'E2a': 2,
 'Foxo1': 3,
 'Pax5': 4,
 'Ikzf1': 5,
 'CD19': 6,
 'BCR': 7,
 'RAG': 8,
 'RUNX1': 9,
 'Vpreb1': 10,
 'EBF1': 11,
 'Proliferation': 12,
 'Apoptosis': 13,
 'Cycle-arrest': 14,
 'CCND3': 15,
 'Bcl2': 16,
 'TSLP': 17,
 'CLP': 18,
 'PreproB': 19,
 'ProB': 20,
 'PreB': 21,
 'Naive B cell': 22,
 'JAK-STAT': 23,
 'Syk': 24,
 'Bcl6': 25,
 'CDKN1A': 26,
 'CDKN1B': 27,
 'Bok': 28,
 'preBCR': 29,
 'Bach2': 30,
 'CXCL12': 31,
 'CXCR4': 32,
 'Id3': 33,
 'Ikzf3': 34,
 'Igll1': 35,
 'CRLF2': 36,
 'Myc': 37,
 'PIP3': 38,
 'CCND2': 39,
 'BCR-ABL': 40,
 'CDKN2A': 41,
 'preBCR_Pax5': 42,
 'AKT': 43,
 'DNA-damage': 44,
 'Bim': 45,
 'p53': 46,
 'MAPK': 47,
 'ERK1_2': 48,
 'Blnk': 49,
 'CXCR4_CXCL12': 50,
 'KO_preBCR': 51,
 'KO_JAK_STAT': 52,
 'IKZF1_ko': 53,
 'BCL6_KO': 54,
 'MYC_KO': 55,
 'KO_CDKN1A': 56,
 'KO_CDKN1B': 57,
 'KO_CDKN2A': 58,
 'Force-Pax5': 59,
 'EBF1_KO': 60,
 'PAX5_KO': 61,
 'KO_PIP3': 62,
 'KO_MAPK': 63,
 'KO_BCRABL': 64}

In [28]:
m['Model']['Variables'][variableDict['Ikzf1']]

{'Name': 'Ikzf1',
 'Id': 17,
 'RangeFrom': 0,
 'RangeTo': 0,
 'Formula': 'min(2,(2+var(8)))-var(408)'}

In [29]:
m['Model']['Variables'][variableDict['Il7']]['RangeTo'] = 2
m['Model']['Variables'][variableDict['Ikzf1']]['RangeTo'] = 2
qM = pybma.model_to_qn(m)
#pM = pybma.check_stability(qM)

#print("###Modified Model###")
#print(pM)

In [30]:
encoder

In [31]:
final_state_tuple = stateToTuple(final_state)
encoder = StateEncoder(m)
variables = list(final_state.keys())
variable_names = [variable_names_lookup[i] for i in variables]

In [32]:
finding, trace = exists_order_before_online(qM, encoder, variables, final_state_tuple, variableDict['PreproB'], 1, variableDict['CLP'], 1)
print(finding)
for state in trace:
    s = tupleToState(state,variable_names)
    print(s['CLP'],s['Naive B cell'])

True
0 0
0 0
0 0


In [33]:
finding, trace = exists_order_before_online(qM, encoder, variables, final_state_tuple, variableDict['Naive B cell'], 1, variableDict['CLP'], 1)

In [34]:
print(finding)

True


In [35]:
check_index_mapping(variables, encoder, variableDict['Naive B cell'], variableDict['CLP'])

Gene i=22 -> '164'
Gene j=18 -> '146'
Encoder max_levels: [2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 4, 4, 4, 2, 2, 0, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 0, 2, 2, 2, 4, 2, 2, 2, 2, 2, 2, 0, 2, 2, 2, 2, 0, 0, 2, 0, 2, 2, 2, 2, 2]
variables list:     [2, 3, 7, 8, 11, 17, 19, 30, 38, 59, 68, 74, 120, 121, 122, 126, 130, 133, 146, 147, 148, 149, 164, 171, 179, 180, 181, 182, 198, 206, 211, 231, 232, 236, 237, 247, 254, 257, 266, 272, 275, 280, 293, 307, 313, 322, 331, 343, 344, 366, 383, 390, 392, 408, 414, 415, 423, 424, 425, 437, 457, 464, 471, 472, 473]


In [36]:
for state in trace:
    s = tupleToState(state,variable_names)
    print(s['CLP'],s['Naive B cell'])

0 0
0 0
0 0


In [37]:
res = list(zip(final_state.keys(),variable_names))
print(res[22])
list(zip(final_state.keys(),variable_names))

(164, 'Naive B cell')


[(2, 'Ilr7'),
 (3, 'Il7'),
 (7, 'E2a'),
 (8, 'Foxo1'),
 (11, 'Pax5'),
 (17, 'Ikzf1'),
 (19, 'CD19'),
 (30, 'BCR'),
 (38, 'RAG'),
 (59, 'RUNX1'),
 (68, 'Vpreb1'),
 (74, 'EBF1'),
 (120, 'Proliferation'),
 (121, 'Apoptosis'),
 (122, 'Cycle-arrest'),
 (126, 'CCND3'),
 (130, 'Bcl2'),
 (133, 'TSLP'),
 (146, 'CLP'),
 (147, 'PreproB'),
 (148, 'ProB'),
 (149, 'PreB'),
 (164, 'Naive B cell'),
 (171, 'JAK-STAT'),
 (179, 'Syk'),
 (180, 'Bcl6'),
 (181, 'CDKN1A'),
 (182, 'CDKN1B'),
 (198, 'Bok'),
 (206, 'preBCR'),
 (211, 'Bach2'),
 (231, 'CXCL12'),
 (232, 'CXCR4'),
 (236, 'Id3'),
 (237, 'Ikzf3'),
 (247, 'Igll1'),
 (254, 'CRLF2'),
 (257, 'Myc'),
 (266, 'PIP3'),
 (272, 'CCND2'),
 (275, 'BCR-ABL'),
 (280, 'CDKN2A'),
 (293, 'preBCR_Pax5'),
 (307, 'AKT'),
 (313, 'DNA-damage'),
 (322, 'Bim'),
 (331, 'p53'),
 (343, 'MAPK'),
 (344, 'ERK1_2'),
 (366, 'Blnk'),
 (383, 'CXCR4_CXCL12'),
 (390, 'KO_preBCR'),
 (392, 'KO_JAK_STAT'),
 (408, 'IKZF1_ko'),
 (414, 'BCL6_KO'),
 (415, 'MYC_KO'),
 (423, 'KO_CDKN1A'),
 (424

In [38]:
variableDict['Naive B cell']

22

In [39]:
final_state

{2: 0,
 3: 0,
 7: 0,
 8: 0,
 11: 0,
 17: 0,
 19: 0,
 30: 0,
 38: 0,
 59: 2,
 68: 0,
 74: 0,
 120: 1,
 121: 1,
 122: 1,
 126: 0,
 130: 0,
 133: 0,
 146: 1,
 147: 0,
 148: 0,
 149: 0,
 164: 0,
 171: 0,
 179: 0,
 180: 0,
 181: 0,
 182: 0,
 198: 0,
 206: 0,
 211: 0,
 231: 2,
 232: 0,
 236: 0,
 237: 0,
 247: 0,
 254: 0,
 257: 0,
 266: 0,
 272: 1,
 275: 0,
 280: 1,
 293: 0,
 307: 0,
 313: 0,
 322: 0,
 331: 0,
 343: 0,
 344: 0,
 366: 0,
 383: 0,
 390: 0,
 392: 0,
 408: 0,
 414: 0,
 415: 0,
 423: 0,
 424: 0,
 425: 0,
 437: 0,
 457: 0,
 464: 0,
 471: 0,
 472: 0,
 473: 0}

In [40]:
final_state_tuple[22]

0

In [41]:
trace[0][22]

0

In [42]:
trace[1][22]

2

In [43]:
i0 = async_step(qM, final_state)

In [44]:
i0s = i0.pop()

In [45]:
i1 = async_step(qM, tupleToState(i0s,variables))

In [46]:
i1s = i1.pop()

In [47]:
print(i0s[22],i1s[22])

0 0
